In [ ]:
# Install only if not already installed
# !pip install torch transformers datasets scikit-learn tqdm

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm

from datasets import load_dataset
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

from transformers import AutoTokenizer, AutoModel

In [ ]:
# Load dataset from HuggingFace
dataset = load_dataset("ag_news")

train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])

print("Training samples:", len(train_df))
print("Test samples:", len(test_df))

train_df.head()

In [3]:
# Create TF-IDF vectorizer
# max_features limits vocabulary size (top 10k frequent words)
vectorizer = TfidfVectorizer(max_features=10000)

# Learn vocabulary from training text and transform it into TF-IDF matrix
X_train_tfidf = vectorizer.fit_transform(train_df['text'])

# Transform test data using same vocabulary
X_test_tfidf = vectorizer.transform(test_df['text'])

# Initialize Logistic Regression classifier
clf = LogisticRegression(max_iter=200)

# Train model
clf.fit(X_train_tfidf, train_df['label'])

# Predict on test data
preds_tfidf = clf.predict(X_test_tfidf)

# Evaluate
print("TF-IDF Accuracy:", accuracy_score(test_df['label'], preds_tfidf))
print(classification_report(test_df['label'], preds_tfidf))

TF-IDF Accuracy: 0.9135526315789474
              precision    recall  f1-score   support

           0       0.93      0.90      0.91      1900
           1       0.95      0.98      0.97      1900
           2       0.88      0.88      0.88      1900
           3       0.89      0.89      0.89      1900

    accuracy                           0.91      7600
   macro avg       0.91      0.91      0.91      7600
weighted avg       0.91      0.91      0.91      7600



## Setup Tokenizer

In [4]:
# Use BERT tokenizer to convert text into token IDs
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

MAX_LEN = 128
BATCH_SIZE = 64
EPOCHS = 3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Custom Dataset Class

In [5]:
from torch.utils.data import Dataset, DataLoader

class NewsDataset(Dataset):
    
    def __init__(self, texts, labels):
        
        # Tokenize entire dataset
        # truncation=True → cut long sentences
        # padding=True → make all same length
        self.encodings = tokenizer(
            texts.tolist(),
            truncation=True,
            padding=True,
            max_length=MAX_LEN
        )
        
        # Store labels separately
        self.labels = labels.tolist()
    
    def __getitem__(self, idx):
        # For a given index, return:
        # input_ids tensor
        # attention_mask tensor
        # label tensor
        
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }
        
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    
    def __len__(self):
        return len(self.labels)

## Create DataLoaders

In [6]:
train_dataset = NewsDataset(train_df['text'], train_df['label'])
test_dataset = NewsDataset(test_df['text'], test_df['label'])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

## Define LSTM Model

In [7]:
class LSTMClassifier(nn.Module):
    
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        
        # Embedding layer → converts token IDs into dense vectors
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        
        # LSTM layer
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        
        # Final classification layer
        self.fc = nn.Linear(hidden_dim, num_classes)
    
    def forward(self, input_ids):
        
        # Convert token IDs to embeddings
        x = self.embedding(input_ids)
        
        # Pass through LSTM
        _, (hidden, _) = self.lstm(x)
        
        # Use last hidden state
        output = self.fc(hidden[-1])
        
        return output

## Train Model

In [8]:
model = LSTMClassifier(
    vocab_size=tokenizer.vocab_size,
    embed_dim=128,
    hidden_dim=128,
    num_classes=4
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    for batch in tqdm(train_loader):
        
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        
        outputs = model(input_ids)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}")

100%|██████████████████████████████████████████████████████████████████████████████| 1875/1875 [02:38<00:00, 11.79it/s]


Epoch 1, Loss: 1.137754060014089


100%|██████████████████████████████████████████████████████████████████████████████| 1875/1875 [02:42<00:00, 11.50it/s]


Epoch 2, Loss: 0.3567442301352819


100%|██████████████████████████████████████████████████████████████████████████████| 1875/1875 [02:52<00:00, 10.87it/s]

Epoch 3, Loss: 0.2441130781273047


## Evaluate LSTM

In [10]:
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids)
        preds = torch.argmax(outputs, dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("LSTM Accuracy:", accuracy_score(all_labels, all_preds))

LSTM Accuracy: 0.9018421052631579


## Model 3 — BERT as Feature Extractor (Frozen)

In [14]:
bert = AutoModel.from_pretrained("bert-base-uncased").to(device)
bert.eval()

def get_bert_embeddings(texts):
    
    embeddings = []
    
    with torch.no_grad():
        for text in tqdm(texts):
            
            inputs = tokenizer(
                text,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=128
            ).to(device)
            
            outputs = bert(**inputs)
            
            # CLS token embedding
            cls_embedding = outputs.last_hidden_state[:, 0, :]
            
            embeddings.append(cls_embedding.squeeze().cpu().numpy())
    
    return np.array(embeddings)

In [15]:
X_train_bert = get_bert_embeddings(train_df['text'][:5000])
y_train_bert = train_df['label'][:5000]

X_test_bert = get_bert_embeddings(test_df['text'][:2000])
y_test_bert = test_df['label'][:2000]

100%|██████████████████████████████████████████████████████████████████████████████| 2000/2000 [03:11<00:00, 10.44it/s]


In [16]:
clf_bert = LogisticRegression(max_iter=300)
clf_bert.fit(X_train_bert, y_train_bert)

preds_bert = clf_bert.predict(X_test_bert)

print("BERT Feature Accuracy:",
      accuracy_score(y_test_bert, preds_bert))

BERT Feature Accuracy: 0.864


C:\Users\admin\.conda\envs\tf\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 300 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=300).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
If accuracy ≈ 92–94%:

Why higher?

Because BERT embeddings are:
Contextual
Pretrained on massive corpus
Already semantically rich

Representation quality matters more than model complexity.